# 5 · Lessons learned: question wording

We checked every Jev question against four prompt anti-patterns, tried a fix for each one we found, and measured it against the old wording on the same data. This notebook recomputes every comparison from the recordings, so it runs offline and costs nothing. The write-up is `docs/lessons-learned.md`.

1. **Jev doesn't know your world.** Put the context it needs in state.
2. **A question ID carries no meaning.** The requirement belongs in `instructions`. Already met everywhere.
3. **Score items one at a time** instead of one Choice over a blob.
4. **Describe the property, not the surface.**

One run each, on 30 DQ rows and 25 source columns: a difference of one column or two rows is a hint, not proof.

In [ ]:
import pandas as pd

from jevdemo import dq, lessons, schema
from jevdemo.judge import MODEL, Judge
from jevdemo.pipeline import HI, LO

judge = Judge()  # JEV_MODE=live for live calls; default replays recordings
print(f"mode: {judge.mode} · model: {MODEL}")

## Schema mapping: the type suffix (rule 4) and the source system (rule 1)
`default` is what `mapping_request` sends now: no `(type)` suffix on option descriptions, and the source system in state. The other columns switch one or both changes back.

In [ ]:
sources, targets, gold = (
    schema.load_sources(),
    schema.load_targets(),
    schema.load_gold(),
)
request = schema.mapping_request(sources[0], targets)
print("state:", request[0]["source_system"])
print("option:", request[1]["target"].criteria["customer.customer_id"])

decided = {
    name: schema.decide(
        [
            {
                **schema.to_mapping(s, a.response),
                "shortlist": schema.shortlist(s, targets),
            }
            for s, a in zip(
                sources,
                judge.ask_many(
                    [schema.mapping_request(s, targets, **kw) for s in sources]
                ),
            )
        ]
    )
    for name, kw in schema.VARIANTS.items()
}
pd.DataFrame(
    {
        name: {
            **schema.evaluate(d, gold),
            "prefilled_wrong": sum(
                m["decision"] == "accept" and m["target_col"] != gold[m["source_col"]]
                for m in d
            ),
        }
        for name, d in decided.items()
    }
).round(2)

The same runs per column: one square per source column, in the same place in every run, so a square that changes colour down a column is a mapping the wording changed. Hover for the pick and the gold target.

In [ ]:
lessons.outcome_grid(lessons.schema_outcomes(judge))

## Schema mapping: one yes/no per target (rule 3)
The last row of the grid asks one Noul per shortlisted target instead of a Choice, with the first version's state, and maps to `none` when no target reaches the 0.80 pre-fill cutoff. Is the strongest yes the right target, and how sure is it?

In [ ]:
scores = lessons.yes_scores(judge)
strongest = scores.loc[scores.groupby("source_col")["p_yes"].idxmax()]
print(
    f"strongest yes is the gold target: {(strongest['lane'] == 'Gold target').sum()} of {strongest['source_col'].nunique()}"
)
lessons.yes_strips(scores)

The ranking is mostly right, but only 3 of 22 gold targets reach the cutoff, so nearly every column falls to `none`: 24% pre-filled. Without a `none` option to compete against, yes/no scores don't sum to one, and here they ran low. This is the cancel split's lesson again: for one pick among exclusive options, keep the Choice.

The columns that moved: what each variant picked, and how sure it was.

In [ ]:
moved = ["ORDER.ORD_BTW_BDR", "ORDER.ORD_LVR_DT", "ORDER.ORD_KLT_NR"]
pd.DataFrame(
    {
        name: {
            m["source_col"]: f"{m['target_col']} {m['p']:.2f} ({m['decision']})"
            for m in d
            if m["source_col"] in moved
        }
        for name, d in decided.items()
    }
).assign(gold=lambda t: t.index.map(gold))

**Redundant surface detail costs confidence in the wrong place.** The shortlist already filters on type, so the suffix told Jev nothing new. Without it, the wrong `ORD_BTW_BDR → none` drops below the 0.80 pre-fill cutoff and goes to review.

**Context in state moved top-1, not pre-fill.** The source system adds one correct top answer (`ORD_LVR_DT`), but pre-fill only rises with both changes together.

## DQ: the address wording (rule 4) and the cancel split (rule 3)
`checks.yaml` is the default; `checks_variants.yaml` keeps the wordings that did worse, under the same ids. Each file gets its own `Cache`: answers are cached by check id, so a shared cache would replay one file's answers for the other. Margin is the gap between the lowest true row and the highest false row.

In [ ]:
rows = dq.load_rows()
truth = {
    "address_is_postal": lambda r: r["is_address"],
    "cancel_reason_code": lambda r: r["cancel_code"] != "other",
}
wordings = {"default": dq.load_checks(), "variant": dq.load_checks(dq.CHECK_VARIANTS)}
p_pass = {}
for name, checks in wordings.items():
    checks = [c for c in checks if c.id in truth]
    judged, _ = dq.judge_checks(rows, checks, judge, dq.Cache())
    p_pass[name] = {(j.check_id, j.value): j.p_pass for j in judged}
    for c in checks:
        print(f"{name:8} {c.id:19} {c.check}")


def separation(name, check_id):
    column = next(c.column for c in wordings[name] if c.id == check_id)
    p = [(p_pass[name][check_id, r[column]], truth[check_id](r)) for r in rows]
    return {
        "min p true": min(x for x, t in p if t),
        "max p false": max(x for x, t in p if not t),
        "review rows": sum(LO < x < HI for x, _ in p),
    }


pd.concat(
    {c: pd.DataFrame({n: separation(n, c) for n in wordings}) for c in truth}
).round(2)

Each dot is one row's p(pass). Rows inside the grey band go to a person; a better wording pushes them out to the edges.

In [ ]:
lessons.margin_strips(lessons.dq_margins(judge))

**The labels define the property, not the rule of thumb.** The tempting fix was to ask about deliverability, but the labels fail a PO box, which receives post fine, and a spot *next to* a house number. The wording that helped says what the street and number have to do: identify one building by its own number. Every labelled row agrees with it:

In [ ]:
labelled = [
    "Postbus 1200, 3000 BE Rotterdam",
    "Molenweg, naast nummer 4",
    "Stationsplein 1",
    "Graaf Florisstraat 3 (achterom)",
    "bel eerst: 010-4123456, Witte de Withstraat 8",
]
label = {r["address"]: r["is_address"] for r in rows}
pd.DataFrame(
    {n: {v: p_pass[n]["address_is_postal", v] for v in labelled} for n in wordings}
).assign(label=lambda t: t.index.map(label)).round(2)

**Splitting a Choice into separate scores isn't a free win.** Rule 3 comes from scoring separate blocks of one input. The cancel check picks one of a few mutually exclusive codes, and the Choice's `other` probability already separated pass from fail. The separate Noul is less sure of plain reasons:

In [ ]:
reasons = sorted({r["cancel_reason"] for r in rows})
table = pd.DataFrame(
    {n: {v: p_pass[n]["cancel_reason_code", v] for v in reasons} for n in wordings}
).assign(
    label=lambda t: t.index.map({r["cancel_reason"]: r["cancel_code"] for r in rows})
)
review = lambda col: col.between(LO, HI, inclusive="neither")  # same bands as bucket()
table[review(table["variant"]) | review(table["default"])].round(2)

## What carries over

- **Measure the margin, not just accuracy.** Every DQ wording decided all its auto-decided rows correctly; only the margin and the review count told them apart.
- **Walk the labels before rewording a check.** A wording that sounds more right can contradict the gold.
- **Add the new wording next to the old one.** Recordings are keyed by the exact request body, so the old wording only stays replayable if its body doesn't change. Before calling the API, the list of unrecorded requests must be exactly the new variants.